In [ ]:
"""
repo_scaffold.py

Scaffolds a generalized ingestion-pipeline repository structure.
Usage:
  python repo_scaffold.py \
    --repo-root client-datalake-pipelines \
    --client myclient --env dev --domain sap \
    --entity orders --source appflow --target s3 --action ingest
"""
import argparse
import sys
from pathlib import Path
from textwrap import dedent

# Templates for generated files
TEMPLATE_README = '''# {repo_name}

This repository contains ingestion pipelines for {client}.

Structure:
- common/: shared modules
- libs/: zipped shared code
- scripts/: pipeline scripts for Glue and other services
- workflows/: Glue Workflow definitions
- infrastructure/: IaC placeholders
- tests/: test stubs
'''

SCRIPT_TEMPLATE = '''"""
{script_filename}

Glue ingestion script placeholder for:
client={client}, domain={domain}, entity={entity}, source={source}, target={target}, action={action}
"""
from common.config import parse_args
from common.logger import get_logger
from common.utils import build_s3_ingest_path

def run():
    args = parse_args()
    # TODO: implement pipeline logic

if __name__ == "__main__":
    run()
'''

TEST_TEMPLATE = '''"""
Test stub for {domain}-{entity} pipeline
"""
from common.utils import build_s3_ingest_path

def test_build_path():
    path = build_s3_ingest_path("{domain}", "{entity}", "2025-01-01")
    assert path.startswith("s3://")
'''

WORKFLOW_TEMPLATE = '''# {domain}_workflow.py

# Glue workflow definition placeholder for {domain}
'''
# Clear sys.argv except for the first element (script name)
sys.argv = ['']  # resetting args so parser doesn't crash in notebook

def main():
    parser = argparse.ArgumentParser(description="Scaffold ingestion pipeline repo structure")
    parser.add_argument("--repo-root", type=str,
                        default="client-datalake-pipelines",
                        help="Path to the repository root to create")
    parser.add_argument("--client", type=str,
                        default="myclient",
                        help="Client name")
    parser.add_argument("--env", type=str,
                        default="dev",
                        choices=["dev", "qa", "prod"],
                        help="Environment")
    parser.add_argument("--domain", type=str,
                        default="sap",
                        choices=["sap", "sparsh"],
                        help="Domain area")
    parser.add_argument("--entity", type=str,
                        default="orders",
                        help="Entity/table name")
    parser.add_argument("--source", type=str,
                        default="appflow",
                        choices=["appflow", "jdbc"],
                        help="Data source")
    parser.add_argument("--target", type=str,
                        default="s3",
                        choices=["s3", "redshift"],
                        help="Target destination")
    parser.add_argument("--action", type=str,
                        default="ingest",
                        choices=["ingesthistory", "ingestcdc"],
                        help="Action type")
    args = parser.parse_args()

    root = Path(args.repo_root)
    if root.exists():
        sys.exit(f"Error: Repository root '{root}' already exists. Aborting.")
    root.mkdir(parents=True)

    # Root README
    (root / 'README.md').write_text(
        TEMPLATE_README.format(repo_name=root.name, client=args.client)
    )

    # common/
    common = root / 'common'
    common.mkdir()
    for fname in ('__init__.py', 'config.py', 'logger.py', 'utils.py'):
        (common / fname).touch()

    # libs/
    libs = root / 'libs'
    libs.mkdir()
    # Create placeholder zip file
    zipname = f"{args.client}-common.zip"
    (libs / zipname).touch()

    # scripts/glue/... pipeline folder
    scripts = root / 'scripts' / 'glue'
    scripts.mkdir(parents=True)
    folder_name = f"{args.domain}_{args.entity}_{args.source}_{args.target}_{args.action}"
    pipeline_dir = scripts / folder_name
    pipeline_dir.mkdir()

    script_filename = f"{args.client}_{folder_name}.py"
    script_file = pipeline_dir / script_filename
    script_file.write_text(
        SCRIPT_TEMPLATE.format(
            script_filename=script_filename,
            client=args.client,
            domain=args.domain,
            entity=args.entity,
            source=args.source,
            target=args.target,
            action=args.action
        )
    )
    # requirements.txt
    (pipeline_dir / 'requirements.txt').touch()

    # workflows/
    workflows = root / 'workflows'
    workflows.mkdir()
    wf_file = workflows / f"{args.domain}_workflow.py"
    wf_file.write_text(WORKFLOW_TEMPLATE.format(domain=args.domain))

    # infrastructure/
    infra = root / 'infrastructure'
    infra.mkdir()
    (infra / 'README.md').write_text("# Infrastructure-as-Code placeholders\n")

    # tests/
    tests = root / 'tests'
    tests.mkdir()
    test_file = tests / f"test_{args.domain}_{args.entity}.py"
    test_file.write_text(
        TEST_TEMPLATE.format(domain=args.domain, entity=args.entity)
    )

    print(f"Scaffolded repository at {root.resolve()}")

if __name__ == '__main__':
    main()


# S3 Bucket Naming Strategy

## Overview

The S3 bucket naming strategy follows the pattern: `{client}-{env}-{zone}`

- `{client}`: Client's name or identifier (e.g., "myclient")
- `{env}`: Environment (e.g., "dev", "staging", "prod")
- `{zone}`: One of the following zones:

| Zone | Purpose |
| --- | --- |
| ingest | Raw incoming data |
| structured | Cleaned and transformed datasets |
| scripts | Glue ETL scripts |
| temp | Temporary or intermediate data |
| archive | Historical backups |
| metadata | Operational metadata, logs |

For example, for client "myclient" in the "dev" environment, the buckets would be:

- `myclient-dev-ingest`
- `myclient-dev-structured`
- `myclient-dev-scripts`
- `myclient-dev-temp`
- `myclient-dev-archive`
- `myclient-dev-metadata`

## Folder Structures

### Ingest Zone

The ingest zone stores raw incoming data, organized by source and entity, with optional partitioning by filter values.

```plaintext
s3://{client}-{env}-ingest/
└── {source}/
    ├── {entity}/
    │   └── filter=filter_value/
    │       └── {entity}_YYYYMMDDHHMMSS.csv
    └── {entity}/
        ├── {entity}_YYYYMMDDHHMMSS.csv
        └── {entity}_YYYYMMDDHHMMSS.csv
```

### Structured Zone

The structured zone contains cleaned and transformed datasets, following a similar structure to the ingest zone.

```plaintext
s3://{client}-{env}-structured/
└── {source}/
    ├── {entity}/
    │   └── filter=filter_value/
    │       └── {entity}_YYYYMMDDHHMMSS.csv
    └── {entity}/
        ├── {entity}_YYYYMMDDHHMMSS.csv
        └── {entity}_YYYYMMDDHHMMSS.csv
```

### Scripts Zone

The scripts zone holds service-specific artifacts such as Glue job scripts, Athena query definitions, and Lambda functions.

```plaintext
s3://{client}-{env}-scripts/
├── glue/
│   ├── {domain}_{entity}_{source}_to_{target}_{action}/
│   │   ├── {client}_{domain}_{entity}_{source}_to_{target}_{action}.py
│   │   └── requirements.txt
│   └── ...
├── athena/
│   ├── {entity}_query.sql
│   └── {another_entity}_ltv_view.sql
└── lambda/
    ├── {function_name}/
    │   ├── handler.py
    │   └── requirements.txt
```

### Temp Zone

The temp zone is used for temporary or intermediate data, including Athena query results and other intermediate files.

```plaintext
s3://{client}-{env}-temp/
├── athena-query-results/
└── intermediate/
```

### Archive Zone

The archive zone stores historical backups, organized similarly to the ingest and structured zones.

```plaintext
s3://{client}-{env}-archive/
└── {source}/
    ├── {entity}/
    │   └── filter=filter_value/
    │       └── {entity}_YYYYMMDDHHMMSS.csv
    └── {entity}/
        ├── {entity}_YYYYMMDDHHMMSS.csv
        └── {entity}_YYYYMMDDHHMMSS.csv
```

### Metadata Zone

The metadata zone contains operational metadata, including run logs, configurations, schemas, and lineage information.

```plaintext
s3://{client}-{env}-metadata/
├── runs/
│   └── {source}/{entity}/
│       └── run_date=YYYY-MM-DD/
│           ├── summary-{runId}.json
│           └── details-{runId}.json
├── configs/
│   ├── source_configs.json
│   └── workflow_configs.json
├── schemas/
│   └── {source}-{entity}-schema.json
└── lineage/
    └── {source}-{entity}-lineage.json
```

In [3]:
import argparse, sys
from datetime import datetime
# Clear sys.argv except for the first element (script name)
sys.argv = ['']  # resetting args so parser doesn't crash in notebook

def generate_s3_naming(client, env, source, entity, domain, target, action):
    # Define the S3 zones
    zones = ["ingest", "structured", "scripts", "temp", "archive", "metadata"]
    
    # Base bucket names
    buckets = {zone: f"{client}-{env}-{zone}" for zone in zones}
    
    # Current timestamp for file examples
    ts = datetime.now().strftime("%Y%m%d%H%M%S")
    date_iso = datetime.now().strftime("%Y-%m-%d")
    
    # Example filter placeholder
    filter_field = "filter"
    filter_value = "filter_value"
    
    # Build example prefixes per zone
    prefixes = {
        "ingest": [
            f"s3://{buckets['ingest']}/{source}/{entity}/",
            f"s3://{buckets['ingest']}/{source}/{entity}/{filter_field}={filter_value}/{entity}_{ts}.csv"
        ],
        "structured": [
            f"s3://{buckets['structured']}/{source}/{entity}/",
            f"s3://{buckets['structured']}/{source}/{entity}/{filter_field}={filter_value}/{entity}_{ts}.csv"
        ],
        "scripts": [
            f"s3://{buckets['scripts']}/glue/{domain}_{entity}_{source}_to_{target}_{action}/",
            f"s3://{buckets['scripts']}/glue/{domain}_{entity}_{source}_to_{target}_{action}/{client}_{domain}_{entity}_{source}_to_{target}_{action}.py"
        ],
        "temp": [
            f"s3://{buckets['temp']}/athena-query-results/",
            f"s3://{buckets['temp']}/intermediate/"
        ],
        "archive": [
            f"s3://{buckets['archive']}/{source}/{entity}/{ts}/"
        ],
        "metadata": [
            f"s3://{buckets['metadata']}/runs/{source}/{entity}/run_date={date_iso}/summary-RUNID.json",
            f"s3://{buckets['metadata']}/runs/{source}/{entity}/run_date={date_iso}/details-RUNID.json",
            f"s3://{buckets['metadata']}/configs/source_configs.json",
            f"s3://{buckets['metadata']}/schemas/{source}-{entity}-schema.json",
            f"s3://{buckets['metadata']}/lineage/{source}-{entity}-lineage.json"
        ]
    }
    
    return buckets, prefixes

def main():
    parser = argparse.ArgumentParser(description="Generate S3 bucket names and example prefixes.")
    parser.add_argument("--client", type=str, default="myclient", help="Client name")
    parser.add_argument("--env", type=str, default="dev", choices=["dev", "qa", "prod"], help="Environment")
    parser.add_argument("--domain", type=str, default="sap", choices=["sap", "sparsh"], help="Domain area")
    parser.add_argument("--entity", type=str, default="orders", help="Entity/table name")
    parser.add_argument("--source", type=str, default="appflow", choices=["appflow", "jdbc"], help="Data source")
    parser.add_argument("--target", type=str, default="s3", choices=["s3", "redshift"], help="Target destination")
    parser.add_argument("--action", type=str, default="ingest", choices=["ingesthistory", "ingestcdc"], help="Action type")
    
    args = parser.parse_args()
    
    buckets, prefixes = generate_s3_naming(
        client=args.client,
        env=args.env,
        source=args.source,
        entity=args.entity,
        domain=args.domain,
        target=args.target,
        action=args.action
    )
    
    print("\nS3 Bucket Names:")
    for zone, name in buckets.items():
        print(f"  {zone}: {name}")
    
    print("\nExample S3 Paths:")
    for zone, paths in prefixes.items():
        print(f"\n{zone.capitalize()} Zone:")
        for p in paths:
            print(f"  {p}")

if __name__ == "__main__":
    main()



S3 Bucket Names:
  ingest: myclient-dev-ingest
  structured: myclient-dev-structured
  scripts: myclient-dev-scripts
  temp: myclient-dev-temp
  archive: myclient-dev-archive
  metadata: myclient-dev-metadata

Example S3 Paths:

Ingest Zone:
  s3://myclient-dev-ingest/appflow/orders/
  s3://myclient-dev-ingest/appflow/orders/filter=filter_value/orders_20250428135352.csv

Structured Zone:
  s3://myclient-dev-structured/appflow/orders/
  s3://myclient-dev-structured/appflow/orders/filter=filter_value/orders_20250428135352.csv

Scripts Zone:
  s3://myclient-dev-scripts/glue/sap_orders_appflow_to_s3_ingest/
  s3://myclient-dev-scripts/glue/sap_orders_appflow_to_s3_ingest/myclient_sap_orders_appflow_to_s3_ingest.py

Temp Zone:
  s3://myclient-dev-temp/athena-query-results/
  s3://myclient-dev-temp/intermediate/

Archive Zone:
  s3://myclient-dev-archive/appflow/orders/20250428135352/

Metadata Zone:
  s3://myclient-dev-metadata/runs/appflow/orders/run_date=2025-04-28/summary-RUNID.json
  s

In [5]:
import argparse, sys
# Clear sys.argv except for the first element (script name)
sys.argv = ['']  # resetting args so parser doesn't crash in notebook

def generate_glue_naming(client, env, domain, entity, source, target, action):
    """
    Generate AWS Glue naming conventions including job, workflow, crawler,
    catalog databases, tables, job folder, and script file name.
    """
    # Glue service names
    glue_job = f"gluejob_{env}_{client}_{domain}_{entity}_{source}_to_{target}_{action}"
    glue_workflow = f"gluewf_{env}_{client}_{domain}_{entity}_{source}_to_{target}"
    glue_crawler = f"gluecr_{env}_{client}_{domain}_{entity}_{source}"

    # Glue job folder and script names
    glue_job_folder = f"{domain}_{entity}_{source}_to_{target}_{action}"
    script_name = f"{client}_{domain}_{entity}_{source}_to_{target}_{action}.py"

    # Catalog databases
    catalog_db_ingest = f"{client}_{env}_ingest"
    catalog_db_structured = f"{client}_{env}_structured"
    catalog_db_metadata = f"{client}_{env}_metadata"

    # Catalog table
    catalog_table = f"{source}_{entity}"

    return {
        "glue_job": glue_job,
        "glue_workflow": glue_workflow,
        "glue_crawler": glue_crawler,
        "glue_job_folder": glue_job_folder,
        "script_name": script_name,
        "catalog_db_ingest": catalog_db_ingest,
        "catalog_db_structured": catalog_db_structured,
        "catalog_db_metadata": catalog_db_metadata,
        "catalog_table": catalog_table
    }

def main():
    parser = argparse.ArgumentParser(description="Generate AWS Glue service names based on conventions.")
    parser.add_argument("--client", type=str,
                        default="myclient",
                        help="Client name")
    parser.add_argument("--env", type=str,
                        default="dev",
                        choices=["dev", "qa", "prod"],
                        help="Environment")
    parser.add_argument("--domain", type=str,
                        default="sap",
                        choices=["sap", "sparsh"],
                        help="Domain area")
    parser.add_argument("--entity", type=str,
                        default="orders",
                        help="Entity/table name")
    parser.add_argument("--source", type=str,
                        default="appflow",
                        choices=["appflow", "jdbc"],
                        help="Data source")
    parser.add_argument("--target", type=str,
                        default="s3",
                        choices=["s3", "redshift"],
                        help="Target destination")
    parser.add_argument("--action", type=str,
                        default="ingest",
                        choices=["ingesthistory", "ingestcdc"],
                        help="Action type")

    args = parser.parse_args()

    names = generate_glue_naming(
        client=args.client,
        env=args.env,
        domain=args.domain,
        entity=args.entity,
        source=args.source,
        target=args.target,
        action=args.action
    )

    print("\nGenerated AWS Glue Service Names:")
    print(f"  Glue Job:         {names['glue_job']}")
    print(f"  Glue Workflow:    {names['glue_workflow']}")
    print(f"  Glue Crawler:     {names['glue_crawler']}")
    print(f"  Glue Job Folder:  {names['glue_job_folder']}")
    print(f"  Script Name:      {names['script_name']}")
    print("\nGlue Catalog Databases:")
    print(f"  Ingest DB:        {names['catalog_db_ingest']}")
    print(f"  Structured DB:    {names['catalog_db_structured']}")
    print(f"  Metadata DB:      {names['catalog_db_metadata']}")
    print("\nGlue Catalog Table:")
    print(f"  Table Name:       {names['catalog_table']}")

if __name__ == "__main__":
    main()



Generated AWS Glue Service Names:
  Glue Job:         gluejob_dev_myclient_sap_orders_appflow_to_s3_ingest
  Glue Workflow:    gluewf_dev_myclient_sap_orders_appflow_to_s3
  Glue Crawler:     gluecr_dev_myclient_sap_orders_appflow
  Glue Job Folder:  sap_orders_appflow_to_s3_ingest
  Script Name:      myclient_sap_orders_appflow_to_s3_ingest.py

Glue Catalog Databases:
  Ingest DB:        myclient_dev_ingest
  Structured DB:    myclient_dev_structured
  Metadata DB:      myclient_dev_metadata

Glue Catalog Table:
  Table Name:       appflow_orders
